# 🤖 Module 12 · Day 1 (Saturday): Build an AI Agent from Scratch
**Designed by Engr Usman Hasan** · Bano Qabil Generative AI course

> **A chatbot answers. An agent acts.**

Bilal, our farm bot, could already *talk* (Modules 02–03), *read the farm's files* (RAG, Modules 08–09) and *learn habits* (fine-tuning, Modules 10–11).
Today he gets **hands**: tools he can choose and use by himself, in a loop, until the job is done.

| Part | What we do | Slides |
|---|---|---|
| 0 | Load a free key (Groq or Gemini) | Setup |
| 1 | See a chatbot fail at a "do something" question | Hook |
| 2 | Make tools (just Python functions!) | Level 2 |
| 3 | Write the ReAct instructions | Level 1 |
| 4 | Be the loop **by hand**: Think → Act → Observe | Level 1 |
| 5 | Put it in a `while` loop: **our first agent** 🎉 | Level 1 |
| 6 | 🏆 Boss level: give the agent a NEW tool | Boss |
| 7 | Break it on purpose (failure lab) | Boss |

**No framework today.** We build the agent with plain Python so you can see there is no magic. On Sunday we use LangChain and LangGraph.

## ⚙️ Part 0: Setup
**Colab:** click the 🔑 **Secrets** icon on the left → **Add new secret** → name `GROQ_API_KEY` (or `GEMINI_API_KEY`) → paste your free key → switch **Notebook access** ON. No GPU needed.

Free keys: [Groq](https://console.groq.com/keys) · [Google AI Studio (Gemini)](https://aistudio.google.com/apikey). **Never share your key or put it on GitHub.**

In [ ]:
# install the two LLM packages (Groq + Gemini) and dotenv (reads the .env file)
%pip install -q "groq==1.7.0" "google-genai==3.6.1" "python-dotenv==1.2.4"

In [ ]:
# ===== STEP 0: load your FREE API key (Colab) =====
# 'os' stores settings (like API keys) in the environment
import os
# Colab Secrets = the 🔑 icon on the left side of Colab
from google.colab import userdata

# choose your brain: "groq" or "gemini"
PROVIDER = "groq"
# the name of the secret we need for that provider
KEY_NAME = "GROQ_API_KEY" if PROVIDER == "groq" else "GEMINI_API_KEY"
# read the key from Colab Secrets and put it in the environment
os.environ[KEY_NAME] = userdata.get(KEY_NAME)
# confirm (we never print the key itself!)
print(f"✅ {KEY_NAME} loaded. Provider: {PROVIDER}")

### The LLM helper
One small function, `ask_llm(messages)`, sends a conversation to Groq **or** Gemini and returns the reply text.
You don't need to understand every line of it today. Just remember: **messages in → text out.**

In [ ]:
# ===== THE LLM HELPER: one small function that talks to Groq OR Gemini =====
# the model we use for each provider (change it here if a name is retired)
MODELS = {"groq": "llama-3.3-70b-versatile", "gemini": "gemini-flash-latest"}

# if we chose Groq...
if PROVIDER == "groq":
    # ...import Groq's Python package
    from groq import Groq
    # create a Groq client with our key
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
# otherwise we use Gemini
else:
    # import Google's Python package
    from google import genai
    # create a Gemini client with our key
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])


# send a list of messages to the LLM and get its reply text back
def ask_llm(messages, stop=None):
    # ----- Groq -----
    if PROVIDER == "groq":
        # send the whole conversation; temperature=0 means "no random answers"
        reply = client.chat.completions.create(
            # which model
            model=MODELS["groq"],
            # the conversation so far
            messages=messages,
            # no randomness
            temperature=0,
            # stop writing when the model reaches one of these words
            stop=stop,
        )
        # take out the text of the reply
        return reply.choices[0].message.content or ""
    # ----- Gemini -----
    # Gemini wants the system message separately...
    system_text = messages[0]["content"]
    # ...and calls the assistant "model" instead of "assistant"
    contents = [{"role": "model" if m["role"] == "assistant" else "user",
                 # each message's text goes inside "parts"
                 "parts": [{"text": m["content"]}]}
                # every message except the first (system) one
                for m in messages[1:]]
    # send it, with the same settings as Groq
    reply = client.models.generate_content(
        # which model
        model=MODELS["gemini"],
        # the conversation
        contents=contents,
        # system message + no randomness + stop words
        config={"system_instruction": system_text, "temperature": 0, "stop_sequences": stop},
    )
    # the reply text ("" if Gemini sent nothing)
    return reply.text or ""

In [ ]:
# quick test: is the LLM answering?
# a conversation with one system message and one user message
test = [{"role": "system", "content": "You are a friendly assistant. Answer in one short line."},
        # the user's message
        {"role": "user", "content": "Say salaam to the Bano Qabil class!"}]
# send it and print the reply
print(ask_llm(test))

## 🙋 Part 1: A chatbot can TALK, but can it DO?
We ask a plain chatbot (no tools) a real farm question. It needs **today's price, today's stock and some maths**.

**Predict first:** will it give the right total? Will it know the stock?

In [ ]:
# the question that needs 3 steps: price + stock + maths
QUESTION = "Ali ko kal 15 litre doodh chahiye. Total bill kitna banega, aur kya itna stock hai?"

# a plain chatbot: only a system message, no tools, no data
chat = [{"role": "system", "content": "You are the assistant of Marhaba Cattle & Dairy Farm in Karachi."},
        # the customer's question
        {"role": "user", "content": QUESTION}]
# ask it and print what it says
print(ask_llm(chat))

**Observe:** the chatbot either **guesses** a price and stock (made up!) or says *"I don't have that information."*
It cannot look anything up. It can only **talk**. To *do* things, it needs **tools** and a **loop**.

*Roman Urdu: Chatbot sirf baat kar sakta hai. Agent kaam kar sakta hai.*

## 🔧 Part 2: Tools are just Python functions
First, the farm's data: prices, stock and rules (a pretend database).

In [ ]:
# ===== THE FARM'S DATA: a tiny pretend database =====
# price of each product in rupees
PRICES = {"milk": 220, "yogurt": 250, "desi ghee": 2600}
# the unit of each product, so answers say "litre" or "kg"
UNITS = {"milk": "litre", "yogurt": "kg", "desi ghee": "kg"}
# how much of each product is in the farm's stock today
STOCK = {"milk": 40, "yogurt": 12, "desi ghee": 3}

# the farm's rules (like the documents we searched in Modules 08 and 09)
FARM_RULES = [
    # rule 1
    "Delivery time: every morning from 6 am to 9 am.",
    # rule 2
    "Delivery areas: DHA, Clifton, Gulshan and PECHS only.",
    # rule 3
    "Delivery charge: Rs 150 per order. Delivery is free when the bill is Rs 3000 or more.",
    # rule 4
    "Bulk discount: 5% off milk when a customer orders 20 litres or more.",
    # rule 5
    "Orders for tomorrow must be placed before 8 pm today.",
    # rule 6
    "Payment: cash on delivery or JazzCash.",
]

# print a quick summary so we can see the data loaded
print("Products:", list(PRICES))
# and how many rules the farm has
print("Rules:", len(FARM_RULES))

Now four tools. Look at the line in `"""triple quotes"""` at the top of each function: that is the tool's **description**.
The LLM never sees our Python code. It only reads the **name + description**, like a waiter reading a menu.

In [ ]:
# ===== THE TOOLS: normal Python functions the agent is allowed to use =====
# Every tool takes ONE text input and returns ONE text answer.
# The first line inside each function (the """docstring""") is the tool's DESCRIPTION.
# The LLM reads ONLY that description to decide which tool to use. Write it clearly!


# tool 1: the price of a product
def get_price(product):
    """Price of ONE farm product. Input: the product name, e.g. milk, yogurt or desi ghee."""
    # make the input small letters and remove extra spaces ("  Milk " -> "milk")
    name = product.lower().strip()
    # if we don't sell it, say so (and list what we DO sell)
    if name not in PRICES:
        # an answer the agent can read and use
        return f"We don't sell '{product}'. We sell: {', '.join(PRICES)}."
    # otherwise give the price with its unit
    return f"{name}: Rs {PRICES[name]} per {UNITS[name]}"


# tool 2: how much is in stock
def check_stock(product):
    """How much of ONE product is in stock today. Input: the product name, e.g. milk."""
    # clean the input the same way
    name = product.lower().strip()
    # unknown product?
    if name not in STOCK:
        # tell the agent clearly
        return f"We don't sell '{product}'. We sell: {', '.join(STOCK)}."
    # the stock amount with its unit
    return f"{name}: {STOCK[name]} {UNITS[name]} in stock"


# tool 3: a calculator (LLMs are bad at maths, Python is perfect at it)
def calculator(expression):
    """Do maths. Input: a maths expression using numbers and + - * / ( ), e.g. 10 * 220."""
    # only these characters are allowed, so nobody can run dangerous code
    allowed = "0123456789+-*/(). "
    # check every character of the input
    if any(ch not in allowed for ch in expression):
        # refuse anything that is not plain maths
        return "Error: only numbers and + - * / ( ) are allowed."
    # try to calculate it
    try:
        # eval() works out the maths; round to 2 decimal places
        return str(round(eval(expression), 2))
    # e.g. dividing by zero or a broken expression
    except Exception as error:
        # report the problem so the agent can try again
        return f"Error: {error}"


# tool 4: search the farm's rules (a mini version of RAG from Modules 08-09)
def search_farm_rules(question):
    """Search the farm's rules (delivery time, delivery areas, delivery charge, discounts, payment). Input: a short question."""
    # split the question into lowercase words longer than 3 letters ("delivery", "charge", ...)
    words = [w.strip("?.,!") for w in question.lower().split() if len(w) > 3]
    # give every rule a score: how many of the question's words appear in it
    scored = [(sum(w in rule.lower() for w in words), rule) for rule in FARM_RULES]
    # keep the 2 best rules that matched at least one word
    best = [rule for score, rule in sorted(scored, reverse=True)[:2] if score > 0]
    # join them with a new line, or say that nothing matched
    return "\n".join(best) if best else "No rule found for that question."


# the agent's TOOLBOX: tool name -> the Python function to run
TOOLS = {
    # name the LLM will write -> real function
    "get_price": get_price,
    # stock checker
    "check_stock": check_stock,
    # calculator
    "calculator": calculator,
    # rule search
    "search_farm_rules": search_farm_rules,
}
# show which tools the agent has
print("Tools ready:", list(TOOLS))

In [ ]:
# test each tool ourselves, BEFORE giving it to the agent
# price of milk
print(get_price("milk"))
# how much yogurt is in stock (capital letters and spaces still work)
print(check_stock("  Yogurt "))
# maths
print(calculator("15 * 220"))
# search the rules
print(search_farm_rules("what is the delivery charge?"))
# a product we don't sell
print(get_price("chicken"))

**Observe:** every tool works **without any AI**. They are normal functions. The agent's job is only to **choose** the right one and **give it the right input**.

Next, the **tool menu**: exactly what the LLM will read.

In [ ]:
# ===== THE TOOL MENU: what the LLM reads to choose a tool =====
# turn the TOOLS dictionary into a menu: one line per tool, "- name: description"
def tool_menu():
    # for every tool, take its name and the docstring we wrote
    return "\n".join(f"- {name}: {func.__doc__}" for name, func in TOOLS.items())

# print the menu, exactly as the LLM will see it
print(tool_menu())

## 🧠 Part 3: The ReAct instructions
**ReAct = Reason + Act.** We ask the LLM to write its thinking in a fixed pattern:

```
Thought: what should I do next?
Action: get_price
Action Input: milk
```
Then it **stops**. *We* run the tool and send back an `Observation:`. It thinks again… until it writes `Final Answer:`.

In [ ]:
# ===== THE ReAct INSTRUCTIONS (the "system prompt") =====
# ReAct = REason + ACT. We tell the LLM to think out loud, then act, then wait.
# {tools} is a blank that gets filled with the tool menu
REACT_TEMPLATE = """You are the helpful agent of Marhaba Cattle & Dairy Farm in Karachi.

You can use these tools:
{tools}

Always reply in EXACTLY this format:
Thought: what you should do next
Action: the name of ONE tool from the list
Action Input: the input for that tool

Then STOP. The system will run the tool and send you an Observation.
You can repeat Thought / Action / Action Input as many times as you need.

When you know the answer, reply:
Thought: I now know the final answer
Final Answer: the answer for the customer

Rules:
- Never guess prices, stock, rules or maths. Use a tool.
- Use only ONE tool at a time.
- If no tool is needed (e.g. a greeting), go straight to Final Answer.
- Reply in the customer's language (English or Roman Urdu)."""


# fill the blank with the CURRENT tool menu (so new tools appear automatically)
def build_system_prompt():
    # .format() puts the menu where {tools} is written
    return REACT_TEMPLATE.format(tools=tool_menu())

# look at the full instructions the agent will get
print(build_system_prompt())

## 🧍 Part 4: Be the loop yourself (one step at a time)
Before writing the loop, **you** play the loop's role. This is exactly what the game on the slides showed.

In [ ]:
# ===== STEP A: THINK (ask the LLM what to do first) =====
# start the conversation: instructions + the customer's question
messages = [{"role": "system", "content": build_system_prompt()},
            # the same question that the chatbot failed
            {"role": "user", "content": QUESTION}]
# ask the LLM; stop=["Observation:"] makes it wait for US to bring the result
reply = ask_llm(messages, stop=["Observation:"])
# print what it is thinking
print(reply)

In [ ]:
# ===== THE PARSER: read the LLM's reply and find what it wants to do =====
# 're' finds patterns in text
import re


# look at one reply and return (kind, value1, value2)
def parse_reply(reply):
    # case 1: the LLM has the final answer
    if "Final Answer:" in reply:
        # everything after "Final Answer:" is the answer
        return "final", reply.split("Final Answer:")[-1].strip(), None
    # case 2: find the line that starts with "Action:"
    action = re.search(r"Action:\s*(.+)", reply)
    # and the line that starts with "Action Input:"
    action_input = re.search(r"Action Input:\s*(.+)", reply)
    # if both were found, the LLM wants to use a tool
    if action and action_input:
        # return the tool name and its input (without extra quotes or spaces)
        return "tool", action.group(1).strip(), action_input.group(1).strip().strip('"')
    # case 3: the reply is in the wrong format
    return "error", None, None

# use the parser on the reply we just got
kind, tool_name, tool_input = parse_reply(reply)
# show what we found
print("kind:", kind, "| tool:", tool_name, "| input:", tool_input)

In [ ]:
# ===== STEP B: ACT (WE run the tool the LLM chose) =====
# find the real function in the toolbox and call it with the LLM's input
observation = TOOLS[tool_name](tool_input)
# this is what the agent will "see"
print("Observation:", observation)

# ===== STEP C: OBSERVE (give the result back to the LLM) =====
# add the LLM's own reply to the conversation
messages.append({"role": "assistant", "content": reply})
# add the tool's result as a new message
messages.append({"role": "user", "content": f"Observation: {observation}"})
# ask the LLM again: what next?
reply = ask_llm(messages, stop=["Observation:"])
# print its next thought
print("\n" + reply)

**Observe:** after seeing the first observation, the LLM decides the **next** action by itself.
Run the STEP B + C cell again (and again) until the reply says `Final Answer:`. Count the loops! 🔁

Doing this by hand is boring. Computers are good at boring. ➜ **a loop.**

## 🔁 Part 5: The agent loop, our first agent
Read the comments: **Think → Act → Observe → repeat**, with a `max_steps` safety limit.

In [ ]:
# ===== THE AGENT LOOP: Think -> Act -> Observe -> repeat =====
# This is the WHOLE secret of an agent: a loop around an LLM.
def run_agent(question, max_steps=6, show=True):
    # the conversation starts with the instructions and the customer's question
    messages = [{"role": "system", "content": build_system_prompt()},
                # the customer's question
                {"role": "user", "content": question}]
    # we save every step here (for printing and for the Streamlit app)
    steps = []
    # print the question
    if show:
        # 🙋 = the customer
        print("🙋 QUESTION:", question)
    # loop at most max_steps times (a safety limit, so it can never loop forever)
    for step in range(1, max_steps + 1):
        # THINK: ask the LLM; stop it before it invents its own "Observation"
        reply = ask_llm(messages, stop=["Observation:"]).strip()
        # read what it wants to do
        kind, value, tool_input = parse_reply(reply)
        # show the LLM's thinking
        if show:
            # the raw Thought / Action lines
            print(f"\n----- step {step} -----\n🧠 {reply}")
        # FINISHED? the LLM gave a final answer
        if kind == "final":
            # remember this step
            steps.append({"step": step, "llm": reply, "observation": None})
            # print the answer
            if show:
                # ✅ = done
                print("\n✅ FINAL ANSWER:", value)
            # stop the loop and return the answer + all steps
            return value, steps
        # ACT: the LLM chose a tool that exists
        if kind == "tool" and value in TOOLS:
            # run the real Python function with the LLM's input
            observation = TOOLS[value](tool_input)
        # the LLM invented a tool name that does not exist
        elif kind == "tool":
            # tell it which tools are real
            observation = f"There is no tool called '{value}'. Use one of: {', '.join(TOOLS)}"
        # the reply was in the wrong format
        else:
            # remind it of the format
            observation = "Wrong format. Reply with Thought / Action / Action Input, or Final Answer."
        # OBSERVE: show what the tool returned
        if show:
            # 👀 = the result the agent sees
            print("👀 Observation:", observation)
        # remember this step
        steps.append({"step": step, "llm": reply, "observation": observation})
        # add the LLM's reply to the conversation...
        messages.append({"role": "assistant", "content": reply})
        # ...and the tool's result, so the LLM can read it in the next loop
        messages.append({"role": "user", "content": f"Observation: {observation}"})
    # we used up all the steps without a final answer
    if show:
        # ⛔ = stopped by the safety limit
        print("\n⛔ Stopped: max_steps reached.")
    # return a polite message + the steps
    return "Sorry, I could not finish. Please ask in a simpler way.", steps

In [ ]:
# TEST 1: an easy question (one tool)
answer, steps = run_agent("Desi ghee kitne ka hai?")

In [ ]:
# TEST 2: the question the chatbot failed (price + stock + maths)
answer, steps = run_agent(QUESTION)
# how many loops did it need?
print("\nLoops used:", len(steps))

In [ ]:
# TEST 3: a rules question + maths (does it apply the bulk discount?)
answer, steps = run_agent("If I order 25 litres of milk, how much will I pay after the bulk discount?")

In [ ]:
# TEST 4: no tool needed. A good agent does not use tools for nothing!
answer, steps = run_agent("Assalam o alaikum! Aap kaun ho?")

**What to notice**
- The **LLM never runs code**. It only *writes* which tool it wants. **Our loop** runs the tool.
- Each question needs a **different number of loops**. The agent decides the steps, not us. That is the difference between an **agent** and a fixed **workflow**.
- `max_steps` is the agent's seatbelt. Without it, a confused agent could loop forever (and spend your free API limit).

*Roman Urdu: Agent = LLM + tools + loop. LLM sochta hai, hamara code kaam karta hai.*

## 🏆 Part 6: Boss level: give the agent a NEW tool
The farm only delivers to some areas. Below is a new tool, `delivery_charge`. Run it, then ask a question that needs it.
**Notice:** we don't change the loop or the instructions. The tool menu updates by itself.

In [ ]:
# ===== YOUR TURN: give the agent a NEW tool =====
# delivery charge for each area (from the farm's rules)
DELIVERY_AREAS = {"dha": 150, "clifton": 150, "gulshan": 150, "pechs": 150}


# tool 5: does the farm deliver to an area, and what does it cost?
def delivery_charge(area):
    """Check if the farm delivers to an area and the delivery charge. Input: the area name, e.g. DHA or Clifton."""
    # clean the input: small letters, first word only ("DHA Phase 6" -> "dha")
    name = area.lower().strip().split()[0]
    # we deliver there
    if name in DELIVERY_AREAS:
        # give the charge and the free-delivery rule
        return f"Yes, we deliver to {area}. Charge: Rs {DELIVERY_AREAS[name]} (free if the bill is Rs 3000 or more)."
    # we don't deliver there
    return f"Sorry, we don't deliver to {area}. We deliver to: DHA, Clifton, Gulshan, PECHS."


# put the new tool in the toolbox (the menu updates by itself!)
TOOLS["delivery_charge"] = delivery_charge
# check that it is there
print("Tools ready:", list(TOOLS))

In [ ]:
# the menu now has 5 tools
print(tool_menu())
# a question that needs the NEW tool + a price + maths
answer, steps = run_agent("I live in Clifton. 10 litre milk ka total delivery ke saath kitna hoga?")

### ✍️ YOUR TURN
1. Write your **own** tool (ideas: `opening_hours(day)`, `product_list(anything)`, `convert_kg_to_grams(kg)`).
2. Give it a **clear one-line description** in triple quotes.
3. Add it: `TOOLS["your_tool_name"] = your_function`
4. Ask a question that needs it, and watch the agent pick it up by itself.

In [ ]:
# ✍️ write your tool here (example to copy and change)
# a tool that tells the farm's opening hours
def opening_hours(day):
    """Farm shop opening hours for a day of the week. Input: a day, e.g. Sunday."""
    # Friday has a prayer break; other days are normal
    if day.lower().strip() == "friday":
        # Friday timing
        return "Friday: 7 am to 12 pm and 3 pm to 9 pm."
    # every other day
    return f"{day}: 7 am to 9 pm."


# add it to the toolbox
TOOLS["opening_hours"] = opening_hours
# test it through the agent
answer, steps = run_agent("Is the farm shop open on Friday afternoon?")

## 💥 Part 7: Failure lab: break it on purpose
Real agents fail. Good engineers know **how** they fail.

In [ ]:
# FAILURE 1: too few steps. The agent is stopped before it can finish.
answer, steps = run_agent(QUESTION, max_steps=1)

In [ ]:
# FAILURE 2: a missing tool. We take the calculator away...
# remove the calculator from the toolbox (and keep it to put back later)
saved = TOOLS.pop("calculator")
# ask a maths question: does it do maths in its head (risky!) or ask for a missing tool?
answer, steps = run_agent("What is the price of 17 litres of milk?")
# put the calculator back
TOOLS["calculator"] = saved

In [ ]:
# FAILURE 3: a bad description. Change the stock tool's description to something useless...
# keep the good description
good = check_stock.__doc__
# a useless description: the LLM can no longer tell what this tool does
check_stock.__doc__ = "A tool."
# ask a stock question and watch the agent's choices
answer, steps = run_agent("How much yogurt do you have today?")
# restore the good description
check_stock.__doc__ = good

**What to notice**
| Failure | What happened | Fix |
|---|---|---|
| `max_steps` too small | Stopped before the answer | Give enough steps, but always keep a limit |
| Missing tool | It guessed, or said it can't | Give it the right tool, and tell it "never guess" |
| Bad description | Wrong tool, or no tool | One clear line: **what** it does + **what input** it needs |

## 🎉 Day 1 complete: Shabash!
You built an agent with **~40 lines** of plain Python: **LLM + tools + loop**.

**Homework before Sunday:** think of an agent for YOUR project. Write down 3 tools it needs (name + one-line description). Bring it on Sunday.

**Tomorrow:** our agent forgets everything after each question (try it!). Sunday we give it **memory**, a **safety check** before it places orders, and rebuild it with **LangChain + LangGraph**.

---
© Module 12 course material designed by **Engr Usman Hasan**.